# Part 1.2: ML Model to Predict Next-Day Price Movement (AAPL)

In this notebook, we train a binary classification model that predicts whether AAPL's closing price will go up or down the next day.

**Steps:**
1. Load the ML-ready dataset from the ETL pipeline
2. Split into training and test sets
3. Train a Logistic Regression model
4. Evaluate performance
5. Export the trained model

In [1]:
import polars as pl
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

DATA_DIR = "../data"

## Step 1: Load ML-Ready Data

In [2]:
df = pl.read_parquet(f"{DATA_DIR}/tickers_ml_ready/aapl_ml_ready.parquet")
print(f"Shape: {df.shape}")
df.head()

Shape: (1217, 16)


Date,Open,High,Low,Close,Volume,Dividend,Shares Outstanding,daily_return,ma_5,ma_20,volatility_20,return_lag_1,return_lag_2,return_lag_3,target
date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i8
2020-05-18,0.0,0.0,0.0,0.002421,0.319426,0.0,1.0,0.560283,0.0,0.0,0.398634,0.401466,0.466632,0.367934,0
2020-05-19,0.002613,0.002818,0.003721,0.0,0.223593,0.0,1.0,0.402011,0.000467,0.003109,0.331923,0.560283,0.401466,0.466632,1
2020-05-20,0.004892,0.004199,0.008609,0.008143,0.251438,0.0,1.0,0.538824,0.00359,0.006109,0.31488,0.402011,0.560283,0.401466,0
2020-05-21,0.007671,0.006078,0.00772,0.004952,0.226326,0.0,1.0,0.392633,0.005557,0.009018,0.320209,0.538824,0.402011,0.560283,1
2020-05-22,0.003613,0.003813,0.006998,0.007647,0.166832,0.0,1.0,0.467777,0.008557,0.011523,0.292093,0.392633,0.538824,0.402011,0


## Step 2: Train/Test Split

We use a time-based split (no shuffling) since this is time series data. The model trains on earlier data and is tested on more recent data.

In [3]:
# Features: everything except Date and target
feature_cols = [c for c in df.columns if c not in ["Date", "target"]]

X = df.select(feature_cols).to_numpy()
y = df["target"].to_numpy()

# Time-based split: 80% train, 20% test (no shuffle)
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

print(f"Train: {X_train.shape[0]} rows")
print(f"Test:  {X_test.shape[0]} rows")

Train: 973 rows
Test:  244 rows


## Step 3: Train Logistic Regression

In [4]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

print("Model trained successfully")

Model trained successfully


## Step 4: Evaluate Performance

In [5]:
y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Down (0)", "Up (1)"]))
print(f"Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.4221

Classification Report:
              precision    recall  f1-score   support

    Down (0)       0.41      0.89      0.56       102
      Up (1)       0.52      0.08      0.15       142

    accuracy                           0.42       244
   macro avg       0.47      0.49      0.35       244
weighted avg       0.48      0.42      0.32       244

Confusion Matrix:
[[ 91  11]
 [130  12]]


## Step 5: Export Model

In [6]:
from pathlib import Path

models_dir = Path("../models")
models_dir.mkdir(exist_ok=True)

model_path = models_dir / "aapl_logistic_regression.joblib"
joblib.dump(model, model_path)
print(f"Model saved to {model_path}")

Model saved to ../models/aapl_logistic_regression.joblib
